# Track B - official retrain, epochs 17-24 (Kaggle, 2 x T4) - rev 3

Resumes **Oriented R-CNN (ConvNeXt-S + PAFPN, Gaussian KLD, ClassBalancedDataset)** at epoch 16 and
trains epochs 17-24 on the **official DOTA v2.0 trainval split** (1,830 images, 18 classes, difficulty
flags preserved), then re-evaluates on the official 7,122-patch validation tiling.

Kaggle gives 2 x T4 and Python 3.11, so the pinned stack (torch 2.0.1 + mmcv 2.0.1 + mmdet 3.1.0)
installs from wheels - no source build, no kernel swap. Cell 2 provisions it into a venv; every step
that needs torch/mmcv runs through that venv.

Recipe unchanged. Only these differ, all forced by the environment:

| Item | Original | Here | Why |
|---|---|---|---|
| GPUs | 1 x A100, then 2 x T4 | 2 x T4 | available hardware |
| per-step batch | 8 (A100) / 4 per GPU (T4) | 1 per GPU x 4 accumulation | effective batch stays 8 |
| work dir | .../track_b_convnext_s_pafpn_kld_2x | .../trackB_official_ep17-24 | the old run is never touched |
| framework | torch 2.0.1, mmcv 2.0.1, mmdet 3.1.0 | identical | pinned venv |

Validation cadence is the original val_interval=2, so epochs 18, 20, 22 and 24 are validated.

Layout: the repo, venv and tiled data live in /tmp (large disk). Checkpoints and deliverables go to
/kaggle/working so they survive the session. /content/data is symlinked to /tmp/data because the
config embedded in epoch_16.pth points there.

In [ ]:
#@title 1 - Environment: GPU, repo, disk, paths { display-mode: "form" }
import subprocess, sys, os
from pathlib import Path

BASE = Path('/tmp/ada')
REPO = BASE / 'ADA'
DATA = Path('/tmp/data')
VENV = Path('/tmp/mmrotate_venv')
WORK = Path('/kaggle/working/work_dirs/trackB_official_ep17-24')
OUT = Path('/kaggle/working/ada_trackB_official')
COMMIT = 'a2ecd69d3d487c3f639c3cec329e0a737454370e'

# Kaggle exports MPLBACKEND=module://matplotlib_inline.backend_inline; that value is
# invalid outside IPython and breaks any child script that touches matplotlib.
os.environ['MPLBACKEND'] = 'Agg'

print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
print('system python', sys.version.split()[0])
print(subprocess.run(['df', '-BG', '/tmp', '/kaggle/working'], capture_output=True, text=True).stdout)

for p in (BASE, DATA, WORK, OUT, OUT / 'logs'):
    p.mkdir(parents=True, exist_ok=True)

# the config inside epoch_16.pth hard-codes /content/data/split_ss_dotav2
Path('/content').mkdir(parents=True, exist_ok=True)
subprocess.run(['rm', '-rf', '/content/data'])
Path('/content/data').symlink_to(DATA)
print('/content/data ->', Path('/content/data').resolve())

if not (REPO / 'scripts').is_dir():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/Lion4467cat/dotav2-obb-benchmark.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', COMMIT], check=True)
print('repo at', subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())
n_gpu = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines())
print('GPUs visible:', n_gpu)
assert n_gpu >= 1, 'no GPU: enable the T4 x2 accelerator'

In [ ]:
#@title 2 - Provision the Python 3.11 venv with the pinned OpenMMLab stack { display-mode: "form" }
import subprocess, time, os
from pathlib import Path

# Kaggle exports MPLBACKEND=module://matplotlib_inline.backend_inline, which is not a
# valid backend outside IPython and makes scripts/verify_env.py fail. Force Agg.
CHILD_ENV = dict(os.environ)
CHILD_ENV['MPLBACKEND'] = 'Agg'

if (VENV / 'bin' / 'python').is_file():
    print('venv already present at', VENV)
else:
    t0 = time.time()
    p = subprocess.Popen(['bash', str(REPO / 'scripts' / 'setup_colab.sh'), str(REPO), str(VENV)],
                         env=CHILD_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    print()
    print('setup exit', p.returncode, 'after', round((time.time() - t0) / 60, 1), 'min')
    assert p.returncode == 0, 'setup_colab.sh failed'
print('venv python', (VENV / 'bin' / 'python').resolve())

In [ ]:
#@title 3 - Verify the venv, stage epoch_16.pth, apply the two mmrotate patches { display-mode: "form" }
import subprocess
from pathlib import Path

PY = VENV / 'bin' / 'python'

PROBE = '''
import torch, mmcv, mmengine, mmdet, os, mmrotate
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.version.cuda)
print('device count', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print('  gpu', i, torch.cuda.get_device_name(i),
          round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1), 'GiB')
print('mmcv', mmcv.__version__, '| mmengine', mmengine.__version__, '| mmdet', mmdet.__version__)
print('mmrotate', os.path.dirname(mmrotate.__file__))
'''
r = subprocess.run([str(PY), '-c', PROBE], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, r.stderr[-2000:]

CKPT16 = Path('/tmp/epoch_16.pth')
if not CKPT16.is_file() or CKPT16.stat().st_size < 700e6:
    print('downloading epoch_16.pth from Drive by file id (gdown)')
    subprocess.run([str(VENV / 'bin' / 'gdown'),
                    '12I5f7mNGUGJpMCIVKBLh1IekMhDkrpPs', '-O', str(CKPT16)], check=True)
print('checkpoint', CKPT16, round(CKPT16.stat().st_size / 1e6), 'MB')
assert CKPT16.stat().st_size > 700e6, 'checkpoint download looks truncated'

# same two source patches the original run used: analytical 2x2 determinant in the
# Gaussian KLD loss (avoids a cuSOLVER error on T4) and chunked rotated-IoU overlaps
PATCH = '''
import sys
from pathlib import Path
sys.path.insert(0, 'REPO')
from scripts.run_kaggle import patch_gaussian_dist_loss, patch_rotate_iou2d_calculator
root = Path('REPO')
patch_gaussian_dist_loss(root)
patch_rotate_iou2d_calculator(root)
'''.replace('REPO', str(REPO))
r = subprocess.run([str(PY), '-c', PATCH], capture_output=True, text=True)
print(r.stdout.strip())
assert r.returncode == 0, r.stderr[-2000:]

In [ ]:
#@title 4 - Official DOTA v2.0 -> 1024/200 tiling, then PROVE it is the right dataset { display-mode: "form" }
import subprocess, time
from collections import Counter
from pathlib import Path

SPLIT = DATA / 'split_ss_dotav2'
RAW = DATA / 'dotav2_raw'
CACHE = DATA / 'raw_archives'
PY = VENV / 'bin' / 'python'

def sh(cmd, cwd=None):
    print('+', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None)
    assert p.returncode == 0, 'command failed: ' + ' '.join(str(c) for c in cmd)

t0 = time.time()
have_patches = (SPLIT / 'train' / 'annfiles').is_dir() and any((SPLIT / 'train' / 'images').glob('*.png'))
import shutil as _sh, os

def _free(paths):
    # Kaggle caps the writable layer: drop every intermediate we no longer need
    # BEFORE the tiler writes ~30 GB of patches, or /tmp flips read-only (EROFS).
    for p in paths:
        try:
            if p.is_dir():
                _sh.rmtree(p, ignore_errors=True)
            elif p.is_file():
                p.unlink()
        except OSError:
            pass

if not have_patches:
    sh([PY, REPO / 'scripts' / 'download_dotav2.py', '--source', 'huggingface',
        '--cache-dir', CACHE, '--out-dir', RAW], cwd=REPO)
    # download_dotav2.py has staged RAW/<split>/{images,labelTxt}; its _extracted
    # tree and raw .tar.gz archives are now redundant - free them immediately.
    _free([RAW / '_extracted', *list(CACHE.rglob('*.tar.gz')), *list(CACHE.rglob('*.zip'))])

    # the v2.0 image tarballs hold ONLY the images new in v2.0 (419 train / 135 val).
    # the full splits are v1.0 + v2.0 = 1830 train + 593 val, so pull the v1.0 images
    # and merge them in before tiling. the v2.0 annotations already cover all of them.
    for split, url in (
        ('train', 'https://huggingface.co/datasets/isaaccorley/dota/resolve/main/dotav1.0_images_train.tar.gz'),
        ('val',   'https://huggingface.co/datasets/isaaccorley/dota/resolve/main/dotav1.0_images_val.tar.gz'),
    ):
        arc = CACHE / split / url.rsplit('/', 1)[-1]
        if not arc.is_file():
            sh(['curl', '-sL', '--retry', '3', '-o', str(arc), url])
        tmp = CACHE / ('_v1_' + split)
        _sh.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        sh(['tar', 'xzf', str(arc), '-C', str(tmp)])
        imgdirs = [p for p in tmp.rglob('images') if p.is_dir()]
        assert imgdirs, 'no images dir found in ' + url
        best = max(imgdirs, key=lambda p: len(list(p.glob('*.png'))))
        (RAW / split / 'images').mkdir(parents=True, exist_ok=True)
        n = 0
        for f in best.glob('*.png'):
            dst = RAW / split / 'images' / f.name
            if not dst.exists():
                # same filesystem under /tmp/data -> hardlink, not a 2nd full copy
                try:
                    os.link(f, dst)
                except OSError:
                    _sh.copy2(f, dst)
                n += 1
        print('v1.0', split, 'images merged:', n, '->', RAW / split / 'images')
        # drop this v1.0 archive + its extracted staging copy before the next split
        _free([tmp, arc])

    # Tile one split at a time using the SAME functions/config as the standalone
    # prepare_dotav2_patches.py, then delete that split's raw imagery once its
    # patches are on disk. This keeps the raw+patch disk peak far below quota.
    _driver = Path('/tmp/tile_one_split.py')
    _driver.write_text(
        'import sys, importlib.util\n'
        'from pathlib import Path\n'
        'spec = importlib.util.spec_from_file_location("_pd", sys.argv[1])\n'
        'pd = importlib.util.module_from_spec(spec); spec.loader.exec_module(pd)\n'
        'raw, splitdir, mmrotate, split = Path(sys.argv[2]), Path(sys.argv[3]), Path(sys.argv[4]), sys.argv[5]\n'
        'src = raw / split; save = splitdir / split\n'
        'import shutil as _sh\n'
        'if save.exists(): _sh.rmtree(save)\n'
        'cfg = pd.build_split_config([str(src / "images")], [str(src / "labelTxt")], save, 1024, 200, 4)\n'
        'pd.run_img_split(mmrotate, cfg)\n'
        'print("[tile]", split, "patches", len(list((save / "images").glob("*.png"))))\n'
    )
    for split in ('train', 'val'):
        sh([PY, _driver, REPO / 'scripts' / 'prepare_dotav2_patches.py', RAW, SPLIT,
            REPO / 'mmrotate', split], cwd=REPO)
        # keep labelTxt (small SOURCE annotations needed by the audit below);
        # the heavy raw imagery is no longer needed once its patches exist.
        _free([RAW / split / 'images'])
print('data ready in', round((time.time() - t0) / 60, 1), 'min')

def scan(ann_dir):
    classes, flags, inst = Counter(), Counter(), 0
    files = list(Path(ann_dir).glob('*.txt'))
    for f in files:
        for line in f.read_text(errors='ignore').splitlines():
            p = line.split()
            if len(p) < 10 or p[0].lower() in ('imagesource:', 'gsd:'):
                continue
            classes[p[8]] += 1
            flags[p[9]] += 1
            inst += 1
    return len(files), inst, classes, flags

V2_CLASSES = {'plane', 'ship', 'storage-tank', 'baseball-diamond', 'tennis-court',
              'basketball-court', 'ground-track-field', 'harbor', 'bridge', 'large-vehicle',
              'small-vehicle', 'helicopter', 'roundabout', 'soccer-ball-field',
              'swimming-pool', 'container-crane', 'airport', 'helipad'}

def ann_root(kind):
    # the tiler consumes RAW/<split>/labelTxt, so that is what must be proven official
    direct = RAW / kind / 'labelTxt'
    if direct.is_dir():
        return direct
    hits = [p for p in RAW.rglob('version2.0') if p.is_dir() and kind in str(p)]
    return hits[0] if hits else None

src_train, src_val = ann_root('train'), ann_root('val')
print('labels that feed the tiler:', src_train, '|', src_val)
tr_f, tr_i, tr_c, tr_fg = scan(src_train)
va_f, va_i, va_c, va_fg = scan(src_val)
print('SOURCE train files', tr_f, 'instances', tr_i, 'classes', len(tr_c))
print('SOURCE val   files', va_f, 'instances', va_i, 'classes', len(va_c))
print('SOURCE train difficulty flags', dict(tr_fg))
print('SOURCE classes', sorted(tr_c))

p_tr_f, p_tr_i, p_tr_c, _ = scan(SPLIT / 'train' / 'annfiles')
p_va_f, p_va_i, p_va_c, p_va_fg = scan(SPLIT / 'val' / 'annfiles')
n_tr_img = len(list((SPLIT / 'train' / 'images').glob('*.png')))
n_va_img = len(list((SPLIT / 'val' / 'images').glob('*.png')))
print()
print('PATCH train', n_tr_img, 'images /', p_tr_f, 'annfiles /', p_tr_i, 'instances')
print('PATCH val  ', n_va_img, 'images /', p_va_f, 'annfiles /', p_va_i, 'instances')
print('PATCH val non-difficult', p_va_fg.get('0', 0), '| difficult(diff!=0)', sum(v for k, v in p_va_fg.items() if k != '0'), '(flag1', p_va_fg.get('1', 0), 'flag2', p_va_fg.get('2', 0), ')')

checks = []
def check(name, ok, detail=''):
    checks.append((name, bool(ok)))
    print(('  PASS  ' if ok else '  FAIL  ') + name + ('   ' + str(detail) if detail else ''))

print()
check('source train annfiles == 1830', tr_f == 1830, tr_f)
check('source train instances == 268627', tr_i == 268627, tr_i)
check('source val annfiles >= 500', va_f >= 500, va_f)
check('class set is exactly DOTA v2.0 (18 classes)', set(tr_c) == V2_CLASSES,
      sorted(V2_CLASSES ^ set(tr_c)) or 'exact match')
check('airport present', 'airport' in tr_c, tr_c.get('airport', 0))
check('helipad present', 'helipad' in tr_c, tr_c.get('helipad', 0))
check('container-crane present', 'container-crane' in tr_c, tr_c.get('container-crane', 0))
check('difficulty flags present at source level',
      tr_fg.get('0', 0) > 0 and tr_fg.get('1', 0) > 0,
      'flag0=' + str(tr_fg.get('0', 0)) + ' flag1=' + str(tr_fg.get('1', 0)))
check('val patches == 7122', n_va_img == 7122, n_va_img)
check('val non-difficult instances == 69542', p_va_fg.get('0', 0) == 69542, p_va_fg.get('0', 0))
check('val difficult instances == 70561 (diff!=0)', sum(v for k, v in p_va_fg.items() if k != '0') == 70561, 'flag1=' + str(p_va_fg.get('1', 0)) + ' flag2=' + str(p_va_fg.get('2', 0)))
check('train patches == 24151 (15749 v1.0 + 8402 v2.0-new)', n_tr_img == 24151, n_tr_img)
check('18 classes at patch level', len(p_tr_c) == 18, len(p_tr_c))

tr_names = {p.stem for p in (SPLIT / 'train' / 'images').glob('*.png')}
va_names = {p.stem for p in (SPLIT / 'val' / 'images').glob('*.png')}
check('no train/val patch overlap', len(tr_names & va_names) == 0, len(tr_names & va_names))

failed = [n for n, ok in checks if not ok]
assert not failed, 'DATASET VERIFICATION FAILED: ' + ', '.join(failed)
print()
print('dataset verification: all', len(checks), 'checks passed')

In [ ]:
#@title 5 - TRAIN epochs 17-24 on 2 x T4 { display-mode: "form" }
import subprocess, time, os
from pathlib import Path

PY = VENV / 'bin' / 'python'
TORCHRUN = VENV / 'bin' / 'torchrun'

# ---- the exact epoch-16 config, straight out of the checkpoint ----
CFG16 = WORK / 'ep16_cfg.py'
if not CFG16.is_file():
    code = ("import torch, pathlib;"
            "ck = torch.load('/tmp/epoch_16.pth', map_location='cpu');"
            "pathlib.Path('" + str(CFG16) + "').write_text(ck['meta']['cfg']);"
            "print('config extracted, chars', len(ck['meta']['cfg']))")
    subprocess.run([str(PY), '-c', code], check=True)

TRAIN_CFG = WORK / 'train_cfg.py'
base = CFG16.read_text()
extra = '''
# ---- official-retrain overrides; the recipe above is untouched ----
work_dir = '%s'
resume = True
load_from = '/tmp/epoch_16.pth'
train_dataloader['batch_size'] = 1
optim_wrapper['accumulative_counts'] = 4
train_cfg['val_interval'] = 2
default_hooks['checkpoint'].update(dict(interval=1, max_keep_ckpts=2, save_best='dota/mAP', out_dir='%s'))
custom_hooks = list(globals().get('custom_hooks', [])) + [
    dict(type='CheckpointHook', interval=1, out_dir='%s', max_keep_ckpts=2),
]
''' % (WORK, WORK, OUT / 'work_dirs' / 'trackB_official_ep17-24')
TRAIN_CFG.write_text(base + extra)
print('training config ->', TRAIN_CFG)

# MMEngine's --resume reads work_dir/last_checkpoint; point it at epoch 16 on a fresh work dir
if not any(WORK.glob('epoch_*.pth')) and not any(WORK.glob('iter_*.pth')):
    (WORK / 'last_checkpoint').write_text('/tmp/epoch_16.pth')
    print('seeded last_checkpoint ->', (WORK / 'last_checkpoint').read_text())

env = dict(os.environ)
env['MPLBACKEND'] = 'Agg'
env['PYTHONPATH'] = str(REPO) + ':' + str(REPO / 'mmrotate') + ':' + env.get('PYTHONPATH', '')
env['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

cmd = [str(TORCHRUN), '--nproc_per_node=2',
       str(REPO / 'mmrotate' / 'tools' / 'train.py'), str(TRAIN_CFG),
       '--work-dir', str(WORK), '--resume', '--launcher', 'pytorch']
print('+', ' '.join(cmd))
print('1 per GPU x 2 GPUs x 4 accumulation = effective batch 8; expected LR at epoch 17 = 2.0366e-05')

LOG = WORK / 'train_stdout.log'
t0 = time.time()
with open(LOG, 'a') as log:
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    p.wait()
elapsed = (time.time() - t0) / 3600
print()
print('train.py exit', p.returncode, 'after', round(elapsed, 2), 'h')
print('checkpoints:', sorted(x.name for x in WORK.glob('*.pth')))
assert p.returncode == 0, 'training failed - see the log above'

In [ ]:
#@title 6 - Final evaluation on the official 7,122-patch val tiling { display-mode: "form" }
import subprocess, json, os, time
from pathlib import Path

PY = VENV / 'bin' / 'python'

cands = sorted(WORK.glob('epoch_*.pth'), key=lambda p: int(p.stem.split('_')[-1]))
assert cands, 'no epoch checkpoint in ' + str(WORK)
CKPT = cands[-1]
print('evaluating', CKPT, round(CKPT.stat().st_size / 1e6), 'MB')

env = dict(os.environ)
env['MPLBACKEND'] = 'Agg'
env['PYTHONPATH'] = str(REPO) + ':' + str(REPO / 'mmrotate') + ':' + env.get('PYTHONPATH', '')
env['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

# the checkpoint's own config carries resume=True, which would make Runner.test()
# try to resume from a last_checkpoint that does not exist here. Eval gets its own copy.
EVAL_CFG = WORK / 'eval_cfg.py'
extra = '''
resume = False
load_from = '%s'
work_dir = '%s'
''' % (CKPT, WORK / 'eval_test')
EVAL_CFG.write_text((WORK / 'ep16_cfg.py').read_text() + extra)
print('eval config ->', EVAL_CFG)

cmd = [str(PY), str(REPO / 'scripts' / 'run_track_eval.py'),
       '--config', str(EVAL_CFG),
       '--work-dir', str(WORK),
       '--checkpoint', str(CKPT),
       '--ann-dir', str(SPLIT / 'val' / 'annfiles'),
       '--batch-size', '1']
print('+', ' '.join(cmd))
t0 = time.time()
p = subprocess.run(cmd, cwd=str(REPO), env=env, capture_output=True, text=True)
out = (p.stdout or '') + (p.stderr or '')
(OUT / 'logs' / 'eval_stdout.log').write_text(out)
print(out[-4000:])
assert p.returncode == 0, 'evaluation failed'

EXT = WORK / 'eval_test' / 'extended_metrics.json'
metrics = json.loads(EXT.read_text())
for k in ('num_images', 'num_gt_non_difficult', 'num_predictions', 'map50', 'ap75', 'map50_95',
          'class_agnostic_ap50', 'classification_gap_ap50'):
    print('  ', k, '=', metrics[k])
print('   operational =', metrics['operational'])
print('   difficulty_recall =', metrics['difficulty_recall'])
print('elapsed', round((time.time() - t0) / 60, 1), 'min')

In [ ]:
#@title 7 - Comparison table, results JSON, deviations { display-mode: "form" }
import json, shutil
from pathlib import Path

metrics = json.loads((WORK / 'eval_test' / 'extended_metrics.json').read_text())

BASELINE = {'classes': ['plane', 'baseball-diamond', 'bridge', 'ground-track-field', 'small-vehicle', 'large-vehicle', 'ship', 'tennis-court', 'basketball-court', 'storage-tank', 'soccer-ball-field', 'roundabout', 'harbor', 'swimming-pool', 'helicopter', 'container-crane', 'airport', 'helipad'], 'track_a_ep10': {'map50': 0.6049, 'ap75': 0.2785, 'map50_95': 0.3146, 'class_agnostic_ap50': 0.7985, 'operational': {'fps': 11.36, 'params_m': None, 'peak_vram_gb': 2.86}, 'per_class': {'plane': {'ap50': 0.9072}, 'baseball-diamond': {'ap50': 0.6984}, 'bridge': {'ap50': 0.3901}, 'ground-track-field': {'ap50': 0.7285}, 'small-vehicle': {'ap50': 0.7653}, 'large-vehicle': {'ap50': 0.7296}, 'ship': {'ap50': 0.8237}, 'tennis-court': {'ap50': 0.9385}, 'basketball-court': {'ap50': 0.6067}, 'storage-tank': {'ap50': 0.8081}, 'soccer-ball-field': {'ap50': 0.5525}, 'roundabout': {'ap50': 0.7123}, 'harbor': {'ap50': 0.6067}, 'swimming-pool': {'ap50': 0.6367}, 'helicopter': {'ap50': 0.5346}, 'container-crane': {'ap50': 0.0}, 'airport': {'ap50': 0.4492}, 'helipad': {'ap50': 0.0}}}, 'track_b_old': {'map50': 0.6968, 'ap75': 0.4347, 'map50_95': 0.4175, 'class_agnostic_ap50': 0.8765, 'operational': {'fps': 5.7, 'params_m': 70.24, 'peak_vram_gb': 4.2}, 'per_class': {'plane': {'ap50': 0.9359}, 'baseball-diamond': {'ap50': 0.7567}, 'bridge': {'ap50': 0.5573}, 'ground-track-field': {'ap50': 0.7983}, 'small-vehicle': {'ap50': 0.822}, 'large-vehicle': {'ap50': 0.8687}, 'ship': {'ap50': 0.9111}, 'tennis-court': {'ap50': 0.957}, 'basketball-court': {'ap50': 0.7621}, 'storage-tank': {'ap50': 0.8286}, 'soccer-ball-field': {'ap50': 0.6723}, 'roundabout': {'ap50': 0.7628}, 'harbor': {'ap50': 0.8027}, 'swimming-pool': {'ap50': 0.6686}, 'helicopter': {'ap50': 0.6685}, 'container-crane': {'ap50': 0.0323}, 'airport': {'ap50': 0.4876}, 'helipad': {'ap50': 0.25}}}}

A = BASELINE['track_a_ep10']
B_OLD = BASELINE['track_b_old']

lines = ['| Metric | Track A ep10 (R50) | Track B OLD ep24 | Track B NEW ep24 | delta NEW-OLD |',
         '|---|---|---|---|---|']
for label, key in (('mAP50', 'map50'), ('AP75', 'ap75'), ('mAP50-95', 'map50_95'),
                   ('class-agnostic AP50', 'class_agnostic_ap50')):
    a, b, n = A[key], B_OLD[key], metrics[key]
    lines.append('| %s | %.4f | %.4f | %.4f | %+.4f |' % (label, a, b, n, n - b))
lines.append('| params (M) | %.2f | %.2f | %.2f | %+.2f |' % (
    A['operational']['params_m'], B_OLD['operational']['params_m'], metrics['operational']['params_m'],
    metrics['operational']['params_m'] - B_OLD['operational']['params_m']))
table = chr(10).join(lines)

per = ['| Class | Track B OLD AP50 | Track B NEW AP50 | delta |', '|---|---|---|---|']
for cls in BASELINE['classes']:
    if cls in metrics['per_class'] and cls in B_OLD['per_class']:
        o = B_OLD['per_class'][cls]['ap50']
        n = metrics['per_class'][cls]['ap50']
        per.append('| %s | %.4f | %.4f | %+.4f |' % (cls, o, n, n - o))
per_table = chr(10).join(per)

comparison = ('# Track B official retrain - comparison' + chr(10) + chr(10) + table +
              chr(10) + chr(10) + '## Per-class AP50' + chr(10) + chr(10) + per_table + chr(10))
(OUT / 'comparison_table.md').write_text(comparison)

per_class = metrics['per_class']
assert len(per_class) == 18, len(per_class)
mean_ap50 = sum(v['ap50'] for v in per_class.values()) / 18
mean_ap = sum(v['ap50_95'] for v in per_class.values()) / 18
print('mean per-class AP50 %.4f vs headline %.4f' % (mean_ap50, metrics['map50']))
print('mean per-class AP50-95 %.4f vs headline %.4f' % (mean_ap, metrics['map50_95']))
assert abs(mean_ap50 - metrics['map50']) < 5e-4
assert abs(mean_ap - metrics['map50_95']) < 5e-4
assert metrics['num_gt_non_difficult'] == 69542

results = {
    'model': 'oriented_rcnn_convnext_s_pafpn_gaussian_kld',
    'checkpoint': 'epoch_24.pth',
    'trained_epochs': '17-24 on official DOTA v2.0 trainval (18 classes, difficulty flags preserved)',
    'resumed_from': 'epoch_16.pth',
    'gpu': '2 x T4 16GB (Kaggle)',
    'per_step_batch': 1,
    'gradient_accumulation': 4,
    'effective_batch': 8,
    'metrics': {k: metrics[k] for k in ('num_images', 'num_gt_non_difficult', 'num_predictions',
                                        'map50', 'ap75', 'map50_95', 'class_agnostic_ap50',
                                        'classification_gap_ap50', 'micro_prf', 'macro_f1_opt',
                                        'scale_ap50', 'orientation_error_deg', 'difficulty_recall',
                                        'operational')},
    'per_class': per_class,
    'baseline_track_b_old': {k: B_OLD[k] for k in ('map50', 'ap75', 'map50_95', 'class_agnostic_ap50')},
    'baseline_track_a_ep10': {k: A[k] for k in ('map50', 'ap75', 'map50_95', 'class_agnostic_ap50')},
}
(OUT / 'results_trackB_official_retrain.json').write_text(json.dumps(results, indent=2))

deviations = [
    '# Deviations from the original recipe',
    '',
    'Every entry below is environment-forced. Model, loss, optimizer, LR schedule, augmentation,',
    'sampler and seed policy are unchanged.',
    '',
    '1. Hardware: 2 x T4 16GB (Kaggle) instead of 1 x A100 then 2 x T4. Effective batch kept at 8 by',
    '   1 sample per GPU with 4-step gradient accumulation.',
    '2. Runtime: a Python 3.11 venv (scripts/setup_colab.sh) is used instead of the notebook kernel,',
    '   so the pinned torch 2.0.1 / mmcv 2.0.1 / mmdet 3.1.0 wheels install without a source build.',
    '3. work_dir is new (trackB_official_ep17-24). The original run directory is read-only input.',
    '4. Validation runs at epochs 18/20/22/24 (original val_interval=2).',
    '5. A second CheckpointHook mirrors each epoch into /kaggle/working so a session end costs at',
    '   most one epoch.',
    '6. The two mmrotate source patches used by the original run are re-applied: analytical 2x2',
    '   determinant in the Gaussian KLD loss, and chunked rotated-IoU overlaps.',
    '',
    'Not changed: model architecture, Gaussian KLD loss, AdamW base LR 1e-4, LinearLR warmup 500',
    'iters, CosineAnnealingLR T_max 24 eta_min 1e-6, ClassBalancedDataset oversample_thr 1e-3,',
    'all augmentation pipelines, seed 1143503445.',
]
(OUT / 'deviations.md').write_text(chr(10).join(deviations))

# keep the training log and config snapshot next to the deliverables
for src in (WORK / 'train_stdout.log', WORK / 'train_cfg.py', WORK / 'ep16_cfg.py'):
    if src.is_file():
        shutil.copy2(src, OUT / src.name)

print()
print(table)
print()
print('wrote:')
for p in sorted(OUT.iterdir()):
    print('  ', p.name)
print()
print('SUMMARY_JSON', json.dumps({'map50': metrics['map50'], 'ap75': metrics['ap75'],
                                 'map50_95': metrics['map50_95'],
                                 'old_map50': B_OLD['map50'], 'old_map50_95': B_OLD['map50_95'],
                                 'fps': metrics['operational'].get('fps'),
                                 'peak_vram_gb': metrics['operational'].get('peak_vram_gb')}))